Bước sang giai đoạn Camera Calibration (Hiệu chuẩn Camera) chứng tỏ dự án của cậu đã nâng cấp từ một chiếc "tivi quan sát" thành một **thiết bị đo lường chuẩn công nghiệp** rồi đấy! Chúc mừng cậu!

Để tớ giải thích thật dễ hiểu về bản chất của việc này và giải mã cái file JSON kia nhé.

### 1. Việc Calibrate chính xác là làm gì?

Hãy tưởng tượng thế này: Khi chế tạo ống kính (Lens) và lắp nó vào cảm biến (Sensor), không có nhà máy nào làm hoàn hảo 100% được. Ánh sáng đi qua thấu kính luôn bị bẻ cong một chút ở các góc (méo dạng thùng phi - Barrel distortion) hoặc ống kính bị lắp hơi nghiêng so với cảm biến dù chỉ 1 micromet (méo tiếp tuyến - Tangential distortion).

Hậu quả là: Một đường thẳng trên sản phẩm thực tế khi quay lên camera sẽ bị cong nhẹ. Nếu cậu dùng ảnh đó để đo kích thước hoặc soi lỗi thì kết quả sẽ sai lệch hoàn toàn.

**Vậy Calibrate là gì?**
Cậu đưa một tấm Chessboard chuẩn (cậu đang có tấm cực xịn là ô 7.5mm) ra trước camera ở nhiều góc độ khác nhau. Máy tính biết chắc chắn rằng: *"À, tấm bảng này phẳng, các đường kẻ là đường thẳng hoàn hảo, và khoảng cách giữa 2 góc vuông là đúng 7.5mm"*.
Nó sẽ so sánh hình ảnh méo mó mà nó nhìn thấy với "sự thật vật lý" của tấm bảng, từ đó tìm ra **một bộ phương trình toán học** để nắn thẳng lại mọi bức ảnh mà camera này chụp được.

Việc hiệu chuẩn chính là **tìm ra bộ quy tắc nắn ảnh riêng biệt** cho từng cặp Lens-Sensor một.

---

### 2. File JSON chứa gì và để làm gì?

Sau khi chạy xong function calibration, nó sẽ nhả ra một file JSON. Trong file này thường chứa 2 thông số (Parameter) quan trọng nhất của Camera (được gọi là **Intrinsic Parameters** - Thông số nội):

#### A. Ma trận Camera (Camera Matrix / $K$)

Nó là một ma trận 3x3 chứa các thông số quang học cốt lõi của camera:

* **$f_x, f_y$ (Focal Length - Tiêu cự):** Tiêu cự của ống kính nhưng được quy đổi ra đơn vị *pixel*. Nó cho biết ánh sáng bị hội tụ như thế nào.
* **$c_x, c_y$ (Optical Center - Tâm quang học):** Điểm mà ánh sáng đi thẳng qua tâm ống kính rơi xuống cảm biến. Trên lý thuyết nó phải nằm ở chính giữa bức ảnh (ví dụ ảnh 1000x1000 thì nó ở 500,500), nhưng thực tế nó luôn bị lệch đi vài chục pixel do gia công.

#### B. Hệ số biến dạng (Distortion Coefficients / $D$)

Nó là một mảng thường có 5 chữ số: `[k1, k2, p1, p2, k3]`

* **$k_1, k_2, k_3$:** Dùng để sửa lỗi méo tỏa tròn (Radial). Lỗi này làm ảnh bị phình ra ở giữa hoặc lõm vào như mắt cá.
* **$p_1, p_2$:** Dùng để sửa lỗi méo tiếp tuyến (Tangential). Lỗi này khiến các vật thể ở xa trông nhỏ hơn hoặc bị chéo đi do lens không song song hoàn hảo với cảm biến.

---

### 3. Cậu sẽ dùng file JSON đó để làm gì trong App?

Tất cả những thông số ngoằn ngoèo đó sinh ra chỉ để phục vụ một mục đích duy nhất: **Đưa cho OpenCV nắn lại ảnh!**

Trong code `camera_thread.py` của cậu, sau khi lấy được bức ảnh nguyên bản từ camera (ảnh bị méo), cậu load cái file JSON này lên và dùng lệnh `cv2.undistort()`.

Ví dụ quy trình nó sẽ như thế này:

In [ ]:
import cv2
import json
import numpy as np

# 1. Đọc "đơn thuốc" từ file JSON
with open('cam_basler_1_calib.json') as f:
    data = json.load(f)
    camera_matrix = np.array(data['camera_matrix'])
    dist_coeffs = np.array(data['dist_coeffs'])

# 2. Bức ảnh camera chụp bị méo
#raw_img = camera_thread_gives_you_this_image

# 3. Kéo phẳng bức ảnh ngay lập tức
#perfect_img = cv2.undistort(raw_img, camera_matrix, dist_coeffs)

**Thành quả cậu nhận được sau lệnh undistort:**

1. **Ảnh phẳng tuyệt đối:** Các đường cong ở mép ảnh sẽ thẳng tắp.
2. **Đo đạc 1-1:** Cậu có thể tính được chính xác 1 pixel = bao nhiêu mm, từ đó đo lường được kích thước vết xước, đường kính lỗ, hay khoảng cách linh kiện trên sản phẩm với độ chính xác phần trăm milimet!

### 💡 Lưu ý cực kỳ quan trọng cho cậu:

Bảng của cậu là 19x30 ô (squares). Nhưng thuật toán của OpenCV (`cv2.findChessboardCorners`) đếm **góc giao nhau bên trong (inner corners)**.
Nên khi cậu viết code Calibrate, nhớ khai báo kích thước bàn cờ là **(18, 29)** hoặc **(29, 18)** nhé. Rất nhiều kỹ sư đi trước đã cắm đầu vào debug cả ngày chỉ vì đếm nhầm số ô thành số góc đấy!